# 01 — Corpus ingest: how raw rows become a build-ready corpus

Blog: [The Road to Cybernaut-1](https://nosible.com/blog/the-road-to-cybernaut-1) —
this notebook covers the **Documents** node at the head of the build-time flow,
everything that happens *before* text processing and embedding.

Nothing here opens a file path. Every read and write goes through the Kedro
**catalog**, which is what makes the same pipeline work against the 63-document
sample and a 100,000-row Hugging Face corpus without a code change.

```mermaid
flowchart TB
    S[("raw_corpus_source<br/>HuggingFaceDataset pinned to a SHA<br/>· or a local JSONL")]
    S --> A["snapshot_raw_corpus"]
    A --> B[("raw_corpus<br/><b>data/01_raw</b><br/>source rows, verbatim")]
    B --> C["normalize_corpus<br/>field_map → Document schema"]
    C --> D[("normalized_documents<br/><b>data/02_intermediate</b><br/>ids, titles, cleaned text")]
    D --> E["select_documents<br/>language · metadata · cap"]
    E --> F[("documents<br/><b>data/03_primary</b><br/>what a build consumes")]
    F -.-> G["index_build pipeline"]
```

**What you will see calculated below**

1. What normalisation actually removes, row by row, and why.
2. That document ids do not depend on row order — the property the whole
   reproducibility story rests on.
3. How `min_text_chars` trades corpus size against document quality.

## Bootstrap

`kedro jupyter lab` injects `catalog` automatically. This helper does the same thing
from *any* kernel — plain Jupyter, VS Code, or the headless executor that runs this
notebook in the test suite — so the file you are reading is the file CI verifies.

In [ ]:
from cybernaut_mini.notebook import kedro_catalog, project_root, run_pipeline

print("project root:", project_root())

catalog = kedro_catalog()
print("catalog entries:", sorted(catalog.keys()))

## A deliberately messy source

Real corpora are not clean. This demo source mirrors the schema the NOSIBLE
Hugging Face datasets publish (`text` / `label` / `netloc` / `url`) and plants one
of each failure mode the normaliser has to survive.

We write it through the catalog entry `raw_corpus_source`, exactly where a
`HuggingFaceDataset` would sit in the `prod` environment.

In [ ]:
GOOD = (
    "Analysts expect refinancing conditions to tighten materially over the coming "
    "quarter as corporate credit spreads widen."
)

source_rows = [
    # 4 ordinary rows, alternating labels so the metadata filter below has bite.
    {"text": f"{GOOD} Case {i}.", "label": "prediction" if i % 2 == 0 else "not-prediction",
     "netloc": "example.com", "url": f"https://example.com/a-{i}"}
    for i in range(4)
] + [
    # Too short — below min_text_chars, dropped.
    {"text": "No comment.", "label": "not-prediction",
     "netloc": "example.com", "url": "https://example.com/short"},
    # Duplicate url of a-0 — collapses to the first occurrence.
    {"text": f"{GOOD} A duplicate of the first row.", "label": "prediction",
     "netloc": "example.com", "url": "https://example.com/a-0"},
    # Messy whitespace — kept, but normalised.
    {"text": "Ragged\n\n   spacing   and\ttabs throughout this otherwise fine passage body.",
     "label": "prediction", "netloc": "example.com", "url": "https://example.com/messy"},
    # No url — still needs a stable id, derived from the body instead.
    {"text": f"{GOOD} This row has no url at all.", "label": "prediction",
     "netloc": "example.com", "url": ""},
]

catalog.save("raw_corpus_source", source_rows)
print(f"wrote {len(source_rows)} source rows through the catalog")
for row in source_rows:
    print(f"  {len(row['text']):>4} chars  url={row['url'] or '(none)':<32} {row['text'][:44]}...")

## Run the acquisition pipeline

Three nodes, three catalog writes — one per layer. Acquisition is a *separate*
pipeline from `index_build` on purpose: fetching is slow and rate-limited and should
happen once, while an index gets rebuilt many times over the same snapshot as
settings are tuned.

In [ ]:
run_pipeline(
    "corpus_ingest",
    corpus={
        "field_map": {"text": "text", "url": "url"},
        "metadata_fields": ["label", "netloc"],
        "default_language": "en",
        "id_prefix": "demo",
        "min_text_chars": 32,
    },
    corpus_selection={"languages": None, "metadata_equals": None, "max_documents": None},
)
print("corpus_ingest complete")

## What each layer holds

Now read all three layers back **through the catalog**. This is the payoff of the
refactor: the layers are addressable by name, so you can inspect any stage without
knowing where it lives on disk.

In [ ]:
raw = catalog.load("raw_corpus")
normalized = catalog.load("normalized_documents")
primary = catalog.load("documents")

print(f"01_raw           raw_corpus            {len(raw):>3} rows   (verbatim source)")
print(f"02_intermediate  normalized_documents  {len(normalized):>3} docs   (Document schema)")
print(f"03_primary       documents             {len(primary):>3} docs   (build-ready)")
print()
print(f"normalisation removed {len(raw) - len(normalized)} of {len(raw)} rows")

### Which rows were dropped, and why

The normaliser skips unusable rows rather than raising on them. That is a deliberate
choice: one malformed row out of 100,000 must not fail a multi-hour production build.
Here we reconstruct the reason for each drop.

In [ ]:
from cybernaut_mini.corpus import CorpusSourceConfig, make_document_id

config = CorpusSourceConfig(
    field_map={"text": "text", "url": "url"},
    metadata_fields=["label", "netloc"],
    id_prefix="demo",
    min_text_chars=32,
)

kept_ids = {doc["id"] for doc in normalized}
seen: set[str] = set()

print(f"{'verdict':<12} {'reason':<22} url")
print("-" * 74)
for row in raw:
    text = " ".join(row["text"].split())
    natural_key = row["url"] or text
    doc_id = make_document_id(config.id_prefix, natural_key)

    if len(text) < config.min_text_chars:
        verdict, reason = "DROPPED", f"< {config.min_text_chars} chars"
    elif doc_id in seen:
        verdict, reason = "DROPPED", "duplicate natural key"
    else:
        verdict, reason = "kept", ""
        seen.add(doc_id)

    print(f"{verdict:<12} {reason:<22} {row['url'] or '(no url)'}")

print()
print(f"survivors: {len(kept_ids)}")

### Whitespace and derived titles

These corpora ship passages with no title, but the shard manifest, the shard summary,
and the `title\ntext` string that gets embedded all assume one exists. A title is
therefore derived from the first sentence, cut on a word boundary.

In [ ]:
for doc in normalized:
    if "Ragged" in doc["text"]:
        print("raw text  :", repr(next(r["text"] for r in raw if "Ragged" in r["text"])))
        print("normalised:", repr(doc["text"]))
        print()

print(f"{'id':<20} {'title':<62} chars")
print("-" * 92)
for doc in normalized:
    print(f"{doc['id']:<20} {doc['title'][:60]:<62} {len(doc['title']):>3}")

## Dynamic 1 — ids do not depend on row order

This is the property everything else rests on. Ids are `blake2b` hashes of the row's
natural key (its `url`, or the body text when there is none), **not** sequential
counters. Re-fetching a corpus whose rows came back in a different order therefore
produces the same ids, the same shard assignments, and the same byte-for-byte index.

Sequential ids would renumber every document and silently invalidate every stored
relevance judgment.

In [ ]:
import random

from cybernaut_mini.corpus import normalize_rows

forward = [d.id for d in normalize_rows(source_rows, config)]

shuffled = list(source_rows)
random.Random(0).shuffle(shuffled)
backward = [d.id for d in normalize_rows(shuffled, config)]

print("original order :", forward[:3], "...")
print("shuffled order :", backward[:3], "...")
print()
print("identical:", forward == backward)
assert forward == backward, "ids must not depend on row order"

## Dynamic 2 — `min_text_chars` trades size against quality

The knob that decides how much of a public corpus survives. Sweeping it shows the
cost directly: too low and boilerplate and navigation text become "documents" that
pollute every shard's keyword profile; too high and you discard real content.

In [ ]:
print(f"{'min_text_chars':>15} {'documents kept':>16} {'mean text length':>18}")
print("-" * 52)
for threshold in (0, 16, 32, 80, 120, 400):
    swept = CorpusSourceConfig(
        field_map={"text": "text", "url": "url"},
        metadata_fields=["label", "netloc"],
        id_prefix="demo",
        min_text_chars=threshold,
    )
    docs = normalize_rows(source_rows, swept)
    mean_len = sum(len(d.text) for d in docs) / len(docs) if docs else 0.0
    print(f"{threshold:>15} {len(docs):>16} {mean_len:>18.1f}")

print()
print("conf/prod uses 120: long enough to exclude nav text and stubs from a web corpus.")

## Dynamic 3 — selection narrows without re-fetching

`select_documents` runs *after* normalisation, between the intermediate and primary
layers. That ordering matters: narrowing the corpus never requires downloading it
again, so you can re-scope a build for free.

In [ ]:
from cybernaut_mini.pipelines.corpus_ingest.nodes import select_documents

print(f"{'selection':<44} {'documents':>10}")
print("-" * 56)
print(f"{'(no filter)':<44} {len(normalized):>10}")

for label, params in [
    ("languages=['en']", {"languages": ["en"]}),
    ("metadata_equals={'label': 'prediction'}", {"metadata_equals": {"label": "prediction"}}),
    ("max_documents=3", {"max_documents": 3}),
]:
    try:
        kept = select_documents(normalized, params)
        print(f"{label:<44} {len(kept):>10}")
    except ValueError as exc:
        print(f"{label:<44} {'ERROR':>10}  {exc}")

## Takeaways

| Observation | Consequence |
|---|---|
| Ids are content-derived, not positional | A re-fetch in a different order still rebuilds the identical index |
| Unusable rows are skipped, not raised on | One bad row in 100,000 cannot fail a multi-hour build |
| Selection sits after normalisation | Re-scoping a corpus costs zero network calls |
| Every read and write went through `catalog` | Swapping the local source for a pinned Hugging Face repo is a `conf/prod` edit, not a code change |

Next: [`02_shard_anatomy.ipynb`](02_shard_anatomy.ipynb) takes the corpus this
produced and asks whether the shards built from it are actually coherent.